# PMLS - Assignment 1
----
**Due Monday by 23:59<br> 
Points 20 Submitting a file upload<br> 
Available 29 Jul at 0:00 - 5 Oct at 23:59**

---

# M10 PMLS - ASSIGNMENT 1
Model Deployment Using FastAPI

 

## Business Context:
A skin clinic conducted a marketing campaign targeting 10,000 customers. The objective is to analyze which customer segments are more likely to respond to the campaign based on demographics and purchase behavior. Insights from this analysis will help improve future targeting strategies and increase campaign effectiveness.

# Tasks
## 1. Gender vs Campaign Response
Create a table showing the response rate (%) for Male and Female customers.


## 2. Age Group vs Campaign Response
Prepare a table showing the response rate (%) across the following age groups:
- <30
- 30–50
- \>50
 

# 3. Purchase in Last Quarter vs Campaign Response
Construct a table showing response rate (%) for customers based on whether they made a purchase in the last quarter (Yes/No).


# 4. Product Usage vs Campaign Response
Categorize customers based on the number of unique products purchased in the last year:

- 1–4
- 5–8
- \>8
Create a table showing the response rate (%) for each category.

# Deployment Task

After completing the analysis:

1. Convert your solution into a FastAPI application.
2. Create an endpoint: /campaign-analysis
3. The endpoint should display results in a tabular format
4. Deploy your API using Render.
5. Share your deployed API link.

In [1]:
import pandas as pd
import statsmodels.api as sm
import plotly.express as px
import numpy as np

campaign_data = pd.read_csv('skin clinic campaign.csv')

print("=====================================")
print("              Head                   ")
print("=====================================")
print(campaign_data.head())
print("=====================================")
print("              Tail                   ")
print("=====================================")
print(campaign_data.tail())
print("=====================================")
print("              Info                   ")
print("=====================================")
print(campaign_data.info())
print("=====================================")
print("              Describe               ")
print("=====================================")
print(campaign_data.describe())
print("=====================================")
print("              DTypes                 ")
print("=====================================")
print(campaign_data.dtypes)

# ================================
# CATEGORICAL
# ================================
cat_cols = ['AgeGroup', 'Gender', 'Purchase_Last_Quarter', 'Unique_Products_Purchased']
masterdata = campaign_data

for col in cat_cols:
    masterdata[col] = masterdata[col].astype('category')
    campaign_data[col] = campaign_data[col].astype('category')

# ================================
# DUMMIES
# ================================
master_dum = pd.get_dummies(masterdata, columns=cat_cols, drop_first=True)
campaign_dum = pd.get_dummies(campaign_data, columns=cat_cols, drop_first=True)

campaign_dum = campaign_dum.reindex(columns=master_dum.columns, fill_value=0)


age_order = ['<30', '30–50', '>50']
if pd.api.types.is_numeric_dtype(campaign_data['AgeGroup']):
    campaign_data['Age_Band'] = pd.cut(campaign_data['AgeGroup'],
                                       bins=[-np.inf, 29, 50, np.inf], labels=age_order)
else:
    campaign_data['Age_Band'] = campaign_data['AgeGroup']   # already banded in the file
    age_order = None
    
y = masterdata['Response_to_Campaign']
if y.dtype == object:
    y = y.map({'Yes': 1, 'No': 0})
y = y.astype(float)
campaign_data['Responded'] = y


X = master_dum.drop(columns=['Response_to_Campaign', 'CustID'])
    



              Head                   
   CustID  Gender AgeGroup Purchase_Last_Quarter  Unique_Products_Purchased  \
0  100001  Female    30-50                   Yes                          9   
1  100002    Male      >50                   Yes                         15   
2  100003    Male      >50                   Yes                         11   
3  100004  Female    30-50                    No                          6   
4  100005  Female      <30                   Yes                         13   

  Response_to_Campaign  
0                  Yes  
1                   No  
2                   No  
3                   No  
4                  Yes  
              Tail                   
      CustID  Gender AgeGroup Purchase_Last_Quarter  \
9995  109996    Male      >50                   Yes   
9996  109997  Female      <30                    No   
9997  109998  Female    30-50                   Yes   
9998  109999    Male    30-50                    No   
9999  110000  Female    

## Tasks 1 - Gender vs Campaign Response 

**Create a table showing the response rate (%) for Male and Female customers.**



In [2]:
def response_rate(data, group_col, order=None, label=None):
    table = (data.groupby(group_col, observed=True)['Responded']
                 .agg(Customers='count', Responders='sum'))
    table['Response_Rate_%'] = (table['Responders'] / table['Customers'] * 100).round(2)
    if order is not None:
        table = table.reindex(order)

    table = table.reset_index().rename(columns={group_col: 'Segment'})
    table['Segment'] = table['Segment'].astype(str)
    
     # First column: 'Age_Band' -> 'Age', 'Gender' -> 'Gender', or use label 
    table.insert(0, 'Response_Segmentation', label or group_col.replace('_Band', ''))
    return table

gender_table   = response_rate(campaign_data, 'Gender')
print(gender_table)


  Response_Segmentation Segment  Customers  Responders  Response_Rate_%
0                Gender  Female       6007      2629.0            43.77
1                Gender    Male       3993      1361.0            34.08


# Task 2 - Age Group vs Campaign Response

**Prepare a table showing the response rate (%) across the following age groups:**

<30
30–50
\>50


In [3]:
age_table      = response_rate(campaign_data, 'Age_Band', order=age_order)
print(age_table)

  Response_Segmentation Segment  Customers  Responders  Response_Rate_%
0                   Age   30-50       4992      2349.0            47.06
1                   Age     <30       2499       823.0            32.93
2                   Age     >50       2509       818.0            32.60


# Task 3 - Purchase in Last Quarter vs Campaign Response

**Construct a table showing response rate (%) for customers based on whether they made a purchase in the last quarter (Yes/No).**





In [4]:
Q_table      = response_rate(campaign_data, 'Purchase_Last_Quarter', order=age_order)
print(Q_table)

   Response_Segmentation Segment  Customers  Responders  Response_Rate_%
0  Purchase_Last_Quarter      No       3486       756.0            21.69
1  Purchase_Last_Quarter     Yes       6514      3234.0            49.65


# Task 4 - Product Usage vs Campaign Response
Categorize customers based on the number of unique products purchased in the last year:

1–4<br>
5–8<br>
\>8<br> 
Create a table showing the response rate (%) for each category.

In [5]:
product_order = ['1–4', '5–8', '>8']
campaign_data['Product_Band'] = pd.cut(campaign_data['Unique_Products_Purchased'],
                                       bins=[0, 4, 8, np.inf], labels=product_order)
product_table  = response_rate(campaign_data, 'Product_Band', order=product_order)
print(product_table)

  Response_Segmentation Segment  Customers  Responders  Response_Rate_%
0               Product     1–4       2091       378.0            18.08
1               Product     5–8       3703      1425.0            38.48
2               Product      >8       4206      2187.0            52.00



#  Deployment Task
After completing the analysis:

1. Convert your solution into a FastAPI application.
2. Create an endpoint: /campaign-analysis
3. The endpoint should display results in a tabular format
4. Deploy your API using Render.
5. Share your deployed API link.

In [6]:
def generate_campaign_analysis_summary():
    print("called generate_campaign_analysis_summary()")
    df = pd.concat([
        response_rate(campaign_data, 'Gender'),
        response_rate(campaign_data, 'Age_Band', order=age_order),
        response_rate(campaign_data, 'Purchase_Last_Quarter', order=['Yes', 'No']),
        response_rate(campaign_data, 'Product_Band', order=product_order),
    ], ignore_index=True)    
    summary = df.round(4).to_dict(orient="records") 
    print(f"returning from generate_campaign_analysis_summary() {summary}")
    return summary 

df = generate_campaign_analysis_summary()
df
    

called generate_campaign_analysis_summary()
returning from generate_campaign_analysis_summary() [{'Response_Segmentation': 'Gender', 'Segment': 'Female', 'Customers': 6007, 'Responders': 2629.0, 'Response_Rate_%': 43.77}, {'Response_Segmentation': 'Gender', 'Segment': 'Male', 'Customers': 3993, 'Responders': 1361.0, 'Response_Rate_%': 34.08}, {'Response_Segmentation': 'Age', 'Segment': '30-50', 'Customers': 4992, 'Responders': 2349.0, 'Response_Rate_%': 47.06}, {'Response_Segmentation': 'Age', 'Segment': '<30', 'Customers': 2499, 'Responders': 823.0, 'Response_Rate_%': 32.93}, {'Response_Segmentation': 'Age', 'Segment': '>50', 'Customers': 2509, 'Responders': 818.0, 'Response_Rate_%': 32.6}, {'Response_Segmentation': 'Purchase_Last_Quarter', 'Segment': 'Yes', 'Customers': 6514, 'Responders': 3234.0, 'Response_Rate_%': 49.65}, {'Response_Segmentation': 'Purchase_Last_Quarter', 'Segment': 'No', 'Customers': 3486, 'Responders': 756.0, 'Response_Rate_%': 21.69}, {'Response_Segmentation': '

[{'Response_Segmentation': 'Gender',
  'Segment': 'Female',
  'Customers': 6007,
  'Responders': 2629.0,
  'Response_Rate_%': 43.77},
 {'Response_Segmentation': 'Gender',
  'Segment': 'Male',
  'Customers': 3993,
  'Responders': 1361.0,
  'Response_Rate_%': 34.08},
 {'Response_Segmentation': 'Age',
  'Segment': '30-50',
  'Customers': 4992,
  'Responders': 2349.0,
  'Response_Rate_%': 47.06},
 {'Response_Segmentation': 'Age',
  'Segment': '<30',
  'Customers': 2499,
  'Responders': 823.0,
  'Response_Rate_%': 32.93},
 {'Response_Segmentation': 'Age',
  'Segment': '>50',
  'Customers': 2509,
  'Responders': 818.0,
  'Response_Rate_%': 32.6},
 {'Response_Segmentation': 'Purchase_Last_Quarter',
  'Segment': 'Yes',
  'Customers': 6514,
  'Responders': 3234.0,
  'Response_Rate_%': 49.65},
 {'Response_Segmentation': 'Purchase_Last_Quarter',
  'Segment': 'No',
  'Customers': 3486,
  'Responders': 756.0,
  'Response_Rate_%': 21.69},
 {'Response_Segmentation': 'Product',
  'Segment': '1–4',
  '

In [7]:
from fastapi import FastAPI
from fastapi.responses import HTMLResponse

# -------------------------------------------------
# Create FastAPI app
# -------------------------------------------------
app = FastAPI()

In [8]:
# -------------------------------------------------
# API endpoint returning JSON data
# -------------------------------------------------
@app.get("/campaign-analysis")
def get_campaign_analysis_summary():
    df  = generate_campaign_analysis_summary()
    return df


# -------------------------------------------------
# HTML frontend embedded directly in endpoint
# -------------------------------------------------
@app.get("/", response_class=HTMLResponse)
def home():

    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Skin Care Campaign Summary</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                margin: 40px;
            }
            button {
                padding: 10px 16px;
                font-size: 16px;
                margin-bottom: 20px;
                cursor: pointer;
            }
            table {
                border-collapse: collapse;
                width: 100%;
            }
            th, td {
                border: 1px solid #ccc;
                padding: 8px;
                text-align: center;
            }
            th {
                background-color: #f4f4f4;
            }
        </style>
    </head>
    <body>

        <h2>Skin Care Campaign Summary</h2>

        <button onclick="loadData()">Load Summary</button>

        <table id="summaryTable">
            <thead>
                <tr>
                    <th>Segment Category</th>
                    <th>Segment</th>                    
                    <th>Customers</th>
                    <th>Responders</th>
                    <th>Response Rate</th>
                </tr>
            </thead>
            <tbody></tbody>
        </table>

        <script>
            function loadData() {
                fetch('/campaign-analysis')
                    .then(response => response.json())
                    .then(data => {
                        const tbody = document.querySelector('#summaryTable tbody');
                        tbody.innerHTML = '';

                        data.forEach(row => {
                            const tr = document.createElement('tr');
                            tr.innerHTML = `
                                <td>${row.Response_Segmentation}</td>
                                <td>${row.Segment}</td>
                                <td>${row.Customers}</td>
                                <td>${row.Responders}</td>
                                <td>${row["Response_Rate_%"]}%</td>
                            `;
                            tbody.appendChild(tr);
                        });
                    })
                    .catch(error => {
                        alert('Error fetching data');
                        console.error(error);
                    });
            }
        </script>

    </body>
    </html>
    """

    return html_content




In [9]:
 home()


'\n    <!DOCTYPE html>\n    <html>\n    <head>\n        <title>Skin Care Campaign Summary</title>\n        <style>\n            body {\n                font-family: Arial, sans-serif;\n                margin: 40px;\n            }\n            button {\n                padding: 10px 16px;\n                font-size: 16px;\n                margin-bottom: 20px;\n                cursor: pointer;\n            }\n            table {\n                border-collapse: collapse;\n                width: 100%;\n            }\n            th, td {\n                border: 1px solid #ccc;\n                padding: 8px;\n                text-align: center;\n            }\n            th {\n                background-color: #f4f4f4;\n            }\n        </style>\n    </head>\n    <body>\n\n        <h2>US Stock Market Summary</h2>\n\n        <button onclick="loadData()">Load Summary</button>\n\n        <table id="summaryTable">\n            <thead>\n                <tr>\n                    <th>Ove